# EAUI 2026 — Predicción de Nivel de Habilidades Digitales

Análisis SHAP para explicabilidad del modelo que predice nivel de habilidades.

In [ ]:
import pyreadstat, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas')


## 1. Carga de Datos

In [ ]:
df, meta = pyreadstat.read_sav("/Users/tomas/github/eaui_subtel/data/sav/2026.sav")
print(f'✓ {df.shape[0]:,} registros × {df.shape[1]} columnas')


## 2. Cálculo de Nivel de Habilidades

In [ ]:
# Definir niveles de habilidades basado en Q8
Q8_cols = [c for c in df.columns if c.startswith('Q8_')]
Q8_basico = ['Q8_10', 'Q8_11', 'Q8_12', 'Q8_13', 'Q8_15', 'Q8_16']
Q8_intermedio = ['Q8_1', 'Q8_2', 'Q8_3', 'Q8_4', 'Q8_5', 'Q8_6', 'Q8_14', 'Q8_17', 'Q8_18']
Q8_avanzado = ['Q8_7', 'Q8_8', 'Q8_9']

def nivel_habilidades(row):
    if row[[c for c in Q8_avanzado if c in df.columns]].eq(1).any():
        return 'Avanzado'
    elif row[[c for c in Q8_intermedio if c in df.columns]].eq(1).any():
        return 'Intermedio'
    elif row[[c for c in Q8_basico if c in df.columns]].eq(1).any():
        return 'Básico'
    else:
        return 'Sin habilidades'

df['nivel_habilidades'] = df.apply(nivel_habilidades, axis=1)

print('✓ Nivel de habilidades calculado:')
print(df['nivel_habilidades'].value_counts().to_dict())


## 3. Recodificaciones y Features

In [ ]:
# Renombrar y recodificar
df = df.rename(columns={'Q1_1':'edad', 'Q1_2':'sexo', 'ZONA':'zona', 'P11':'pago_int', 'Q7_4':'pago_mov', 'Q10':'frecuencia', 'Q11':'horas'})
df['sexo'] = df['sexo'].map({1:'H', 2:'M'})
df['zona'] = df['zona'].map({1:'U', 2:'R'})
df['uso_sp'] = df['Q7'].map({1:1, 2:0})
df['freq'] = df['frecuencia'].map({1:4, 2:3, 3:2, 4:1}).fillna(2)
df['horas_d'] = df['horas'].map({1:0.5, 2:1.5, 3:3, 4:5}).fillna(1)

# Features de ingeniería
P3_cols = [c for c in df.columns if c.startswith('P3_')][:6]
df['n_disp'] = df[[c for c in P3_cols if c in df.columns]].fillna(0).sum(axis=1)

Q21_cols = [c for c in df.columns if c.startswith('Q21_')][:6]
df['n_act'] = df[[c for c in Q21_cols if c in df.columns]].fillna(0).sum(axis=1)

df['intens'] = (df['freq'] * df['horas_d']).fillna(1)

print('✓ Features de ingeniería creados')


## 4. Entrenamiento del Modelo

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Definir features deseados
nf_desired = ['edad', 'n_disp', 'n_act', 'intens', 'pago_int', 'pago_mov']
cf_desired = ['sexo', 'zona', 'uso_sp']

# Filtrar solo columnas que existan en df
nf = [col for col in nf_desired if col in df.columns]
cf = [col for col in cf_desired if col in df.columns]

print(f'Features numéricos disponibles: {nf}')
print(f'Features categóricos disponibles: {cf}')

dm = df[nf + cf + ['nivel_habilidades']].dropna(subset=['nivel_habilidades']).copy()

for col in nf: 
    dm[col] = dm[col].fillna(dm[col].median())
for col in cf: 
    dm[col] = dm[col].fillna('U')

le = {}
for col in cf:
    le_obj = LabelEncoder()
    dm[col+'e'] = le_obj.fit_transform(dm[col].astype(str))
    le[col] = le_obj

af = nf + [c+'e' for c in cf]
print(f'Features finales: {af} (total: {len(af)})')

X, y = dm[af].values, dm['nivel_habilidades'].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
Xtrs, Xtes = sc.fit_transform(Xtr), sc.transform(Xte)

print(f'✓ Dataset: {len(dm):,} registros, {X.shape[1]} features')

rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(Xtrs, ytr)

ypr = rf.predict(Xtes)
acc = accuracy_score(yte, ypr)

print(f'✓ Accuracy: {acc:.4f}')
print(f'\nClases: {list(rf.classes_)}')
print(classification_report(yte, ypr, target_names=rf.classes_))

## 5. Feature Importance

In [ ]:
fi = pd.DataFrame({'Feature': af, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
print('Top Features:')
print(fi.head(8).to_string(index=False))

cm = confusion_matrix(yte, ypr)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=rf.classes_, yticklabels=rf.classes_, ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Matriz de Confusión', fontweight='bold')

tf = fi.head(8)
axes[1].barh(range(len(tf)), tf['Importance'].values, color='steelblue', edgecolor='black')
axes[1].set_yticks(range(len(tf)))
axes[1].set_yticklabels(tf['Feature'].values, fontsize=9)
axes[1].set_title('Top 8 Features', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()
print('✓ Gráficos generados')


## 6. Análisis SHAP

In [ ]:
import shap

print('Calculando SHAP...')
explainer = shap.TreeExplainer(rf)
shap_vals_all = explainer.shap_values(Xtes)

# Para multiclase, shap_vals_all tiene shape (n_samples, n_features, n_classes)
idx_maj = np.argmax([np.sum(yte == c) for c in rf.classes_])
clase = rf.classes_[idx_maj]

# Indexar correctamente: shap_vals_all[:, :, class_idx]
shap_vals = np.abs(shap_vals_all[:, :, idx_maj]).mean(axis=0)

shap_rank = pd.DataFrame({'Feature': af, 'SHAP': shap_vals}).sort_values('SHAP', ascending=False)

print(f'✓ SHAP calculado para clase mayoritaria: {clase}')
print(f'✓ Features analizados: {len(shap_rank)}')
print('\nTop Features por SHAP:')
print(shap_rank.to_string(index=False))

shap_rank.to_csv('shap_ranking_habilidades.csv', index=False)
print('\n✓ shap_ranking_habilidades.csv guardado')

## 7. Comparación SHAP vs Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

tf = fi.head(8)
axes[0].barh(range(len(tf)), tf['Importance'].values, color='steelblue', edgecolor='black')
axes[0].set_yticks(range(len(tf)))
axes[0].set_yticklabels(tf['Feature'].values, fontsize=9)
axes[0].set_title('Feature Importance', fontweight='bold')
axes[0].invert_yaxis()

ts = shap_rank.head(8)
axes[1].barh(range(len(ts)), ts['SHAP'].values, color='coral', edgecolor='black')
axes[1].set_yticks(range(len(ts)))
axes[1].set_yticklabels(ts['Feature'].values, fontsize=9)
axes[1].set_title('SHAP Importance', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()
print('✓ Comparación completada')


## 8. Resumen

In [ ]:
print('\n' + '='*70)
print('ANÁLISIS SHAP - PREDICCIÓN NIVEL DE HABILIDADES')
print('='*70)

print(f'\n📊 DATASET: {len(df):,} → {len(dm):,} registros')
print(f'✓ Accuracy: {acc:.4f}')
print(f'✓ Clases: {list(rf.classes_)}')
print(f'✓ Features: {len(af)}')

print(f'\n🔝 TOP 5 SHAP:')
for i, row in shap_rank.head(5).iterrows():
    print(f'   {i+1}. {row["Feature"]:<15} {row["SHAP"]:.6f}')

print(f'\n✅ ARCHIVOS: eaui2026_v5.ipynb, shap_ranking_habilidades.csv')
print('='*70)
